# SLiMIA Dataset Analysis & EDA

Comprehensive exploratory analysis of the **SLiMIA** (Spheroid Light Microscopy Image Atlas) dataset.
Covers:
- Dataset structure & metadata distributions
- Class balance across all 9 protocol attributes
- Technical replicate split analysis (train / val / test)
- Temporal coverage per experimental group
- Sample image grid across microscopes
- Attribute co-occurrence heatmaps
- Missing data audit

> Runs on Kaggle out of the box. Update `CSV_PATH` and `IMG_DIR` for local use.

In [ ]:
# !pip install tifffile --quiet   # usually pre-installed on Kaggle

In [ ]:
import os
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm import tqdm

sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

# Paths
CSV_PATH   = "/kaggle/input/slimia-metadata/slimia_metadata.csv"
OUTPUT_DIR = "./eda_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Technical replicate splits (as used in all downstream models)
TRAIN_REPS = ["T1", "T2", "T3", "T4"]
VAL_REPS   = ["T5", "T8"]
TEST_REPS  = ["T6", "T7"] + [f"T{i}" for i in range(9, 25)]

LABEL_COLS = [
    "microscope", "cell_line", "culture_medium", "formation_method",
    "seeding_density", "timepoint", "biological_rep", "magnification"
]

## 1. Load & Basic Info

In [ ]:
df = pd.read_csv(CSV_PATH)
df["full_path"] = df["full_path"].astype(str).str.strip()

# Parse numeric timepoint (e.g. "024h" → 24)
def parse_hour(s):
    m = re.search(r"(\d+)", str(s))
    return int(m.group(1)) if m else np.nan

df["timepoint_hour"] = df["timepoint"].apply(parse_hour)

# Assign split label
def assign_split(rep):
    if rep in TRAIN_REPS: return "train"
    if rep in VAL_REPS:   return "val"
    return "test"

df["split"] = df["technical_rep"].apply(assign_split)

print(f"Total images : {len(df):,}")
print(f"Columns      : {df.columns.tolist()}")
print(f"\nSplit counts:")
print(df["split"].value_counts().to_string())
df.head(3)

In [ ]:
df["full_path"] = df["full_path"].str.replace(
    "/kaggle/input/slimia/",
    "/kaggle/input/datasets/ayushsri26108/slimia/",
    regex=False
)

## 2. Missing Data Audit

In [ ]:
missing = df[LABEL_COLS + ["full_path"]].isnull().sum().rename("missing_count")
missing_pct = (missing / len(df) * 100).rename("missing_%")
audit = pd.concat([missing, missing_pct], axis=1)
print(audit.to_string())

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(audit.index, audit["missing_%"], color="salmon")
ax.set_ylabel("Missing (%)")
ax.set_title("Missing Data Per Column")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/missing_data.png")
plt.show()

## 3. Class Distribution — All 8 Attributes

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(18, 20))
for ax, col in zip(axes.flat, LABEL_COLS):
    counts = df[col].astype(str).value_counts()
    n_cls  = len(counts)
    # bar plot (cap at 40 classes for readability)
    top = counts.head(40)
    sns.barplot(x=top.index, y=top.values, ax=ax, palette="Blues_r")
    ax.set_title(f"{col}  ({n_cls} classes)", fontsize=11)
    ax.set_xlabel("")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=90, labelsize=7)
    if n_cls > 40:
        ax.set_title(f"{col}  ({n_cls} classes, top 40 shown)", fontsize=10)

plt.suptitle("Class Distribution Across Protocol Attributes", y=1.01, fontsize=14)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/class_distributions.png", bbox_inches="tight")
plt.show()

## 4. Train / Val / Test Split Balance

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, col in zip(axes.flat, LABEL_COLS):
    # top-N classes
    top_cls = df[col].astype(str).value_counts().head(15).index
    sub = df[df[col].astype(str).isin(top_cls)]
    pivot = sub.groupby([col, "split"]).size().unstack(fill_value=0)
    pivot = pivot.reindex(columns=["train", "val", "test"], fill_value=0)
    pivot.plot(kind="bar", ax=ax, colormap="Set2", legend=(col == LABEL_COLS[0]))
    ax.set_title(f"{col} (top 15)", fontsize=9)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=90, labelsize=6)

plt.suptitle("Split Balance Across Attributes (top-15 classes)", y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/split_balance.png", bbox_inches="tight")
plt.show()

## 5. Images Per Technical Replicate

In [ ]:
rep_counts = df["technical_rep"].value_counts().sort_index()
colors = ["steelblue" if r in TRAIN_REPS
          else ("orange" if r in VAL_REPS else "tomato")
          for r in rep_counts.index]

fig, ax = plt.subplots(figsize=(16, 4))
ax.bar(rep_counts.index, rep_counts.values, color=colors)
ax.set_xlabel("Technical Replicate")
ax.set_ylabel("Image Count")
ax.set_title("Images per Technical Replicate  (blue=train | orange=val | red=test)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/replicate_counts.png")
plt.show()

print("\nReplicate-level summary:")
print(rep_counts.describe().to_string())

## 6. Temporal Coverage — Timepoint Distribution

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 4))

# Histogram
ax1.hist(df["timepoint_hour"].dropna(), bins=50, color="steelblue", edgecolor="white")
ax1.set_xlabel("Timepoint (hours)")
ax1.set_ylabel("Count")
ax1.set_title("Timepoint Distribution")

# Top 20 timepoints
top_tp = df["timepoint"].value_counts().head(20)
sns.barplot(x=top_tp.index, y=top_tp.values, ax=ax2, palette="viridis")
ax2.set_xlabel("Timepoint label")
ax2.set_ylabel("Count")
ax2.set_title("Top 20 Timepoints by Frequency")
ax2.tick_params(axis="x", rotation=90)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/timepoint_distribution.png")
plt.show()

## 7. Longitudinal Sequence Lengths

In [ ]:
# Group by experimental conditions — how many timepoints per group?
group_keys = ["microscope","cell_line","culture_medium","formation_method",
               "seeding_density","magnification","biological_rep","technical_rep"]

seq_lengths = df.groupby(group_keys)["timepoint_hour"].count().reset_index(name="seq_len")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(seq_lengths["seq_len"], bins=40, color="mediumseagreen", edgecolor="white")
axes[0].set_xlabel("Sequence length (timepoints per group)")
axes[0].set_ylabel("Number of groups")
axes[0].set_title("Distribution of Longitudinal Sequence Lengths")

# Cumulative — how many groups have ≥ N timepoints
thresh = np.arange(1, seq_lengths["seq_len"].max() + 1)
cumul  = [(seq_lengths["seq_len"] >= t).sum() for t in thresh]
axes[1].plot(thresh, cumul, color="darkorange")
axes[1].axvline(3, color="red", linestyle="--", label="≥3 (temporal modelling threshold)")
axes[1].set_xlabel("Min sequence length")
axes[1].set_ylabel("Groups with ≥ N frames")
axes[1].set_title("Cumulative: Groups Usable for Temporal Prediction")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/sequence_lengths.png")
plt.show()

print(f"Total experimental groups  : {len(seq_lengths):,}")
print(f"Groups with ≥3 timepoints  : {(seq_lengths['seq_len'] >= 3).sum():,}")
print(f"Groups with ≥5 timepoints  : {(seq_lengths['seq_len'] >= 5).sum():,}")
print(f"Median sequence length     : {seq_lengths['seq_len'].median():.1f}")

## 8. Attribute Co-occurrence Heatmaps

In [ ]:
# Show co-occurrence between key attribute pairs
pairs = [
    ("microscope",     "magnification"),
    ("cell_line",      "culture_medium"),
    ("formation_method","seeding_density"),
    ("microscope",     "cell_line"),
]

fig, axes = plt.subplots(2, 2, figsize=(20, 14))
for ax, (c1, c2) in zip(axes.flat, pairs):
    ct = pd.crosstab(df[c1].astype(str), df[c2].astype(str))
    # Cap rows/cols for readability
    top_r = ct.sum(axis=1).nlargest(20).index
    top_c = ct.sum(axis=0).nlargest(20).index
    ct    = ct.loc[ct.index.isin(top_r), ct.columns.isin(top_c)]
    sns.heatmap(ct, ax=ax, cmap="YlOrRd", fmt="d",
                annot=(ct.shape[0] <= 12 and ct.shape[1] <= 12),
                linewidths=0.3, linecolor="white")
    ax.set_title(f"{c1}  ×  {c2}", fontsize=11)
    ax.tick_params(axis="x", rotation=90, labelsize=7)
    ax.tick_params(axis="y", rotation=0, labelsize=7)

plt.suptitle("Attribute Co-occurrence (count of image pairs)", y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/cooccurrence_heatmaps.png", bbox_inches="tight")
plt.show()

## 9. Class Coverage Across Splits

In [ ]:
print("Class coverage per attribute across splits:\n")
rows = []
for col in LABEL_COLS:
    all_cls   = set(df[col].astype(str).unique())
    train_cls = set(df[df["split"]=="train"][col].astype(str).unique())
    val_cls   = set(df[df["split"]=="val"][col].astype(str).unique())
    test_cls  = set(df[df["split"]=="test"][col].astype(str).unique())
    unseen_val  = val_cls  - train_cls
    unseen_test = test_cls - train_cls
    rows.append({
        "Attribute": col,
        "Total classes": len(all_cls),
        "Train classes": len(train_cls),
        "Val classes":   len(val_cls),
        "Test classes":  len(test_cls),
        "Unseen in val": len(unseen_val),
        "Unseen in test": len(unseen_test),
    })

coverage_df = pd.DataFrame(rows).set_index("Attribute")
print(coverage_df.to_string())
coverage_df.to_csv(f"{OUTPUT_DIR}/class_coverage.csv")

## 10. Sample Image Grid

In [ ]:
def load_thumb(path, size=128):
    """Load a TIFF as a normalised grayscale thumbnail."""
    try:
        arr = tifffile.imread(path).astype(np.float32)
        if arr.ndim == 3:
            if arr.shape[0] in [1, 3] and arr.shape[0] < arr.shape[-1]:
                arr = np.transpose(arr, (1, 2, 0))
            arr = arr.mean(axis=-1)
        arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
        img = Image.fromarray((arr * 255).astype(np.uint8)).resize((size, size))
        return np.array(img)
    except Exception:
        return np.zeros((size, size), dtype=np.uint8)


# One sample image per microscope type
microscopes = df["microscope"].unique()
n_mic       = len(microscopes)
fig, axes   = plt.subplots(2, max(1, (n_mic + 1) // 2), figsize=(3 * max(1, (n_mic+1)//2), 6))
axes        = np.array(axes).flat

for ax, mic in zip(axes, sorted(microscopes)):
    row = df[df["microscope"] == mic].iloc[0]
    img = load_thumb(row["full_path"])
    ax.imshow(img, cmap="gray")
    ax.set_title(f"{mic}", fontsize=8)
    ax.axis("off")

for ax in axes:   # hide unused
    ax.axis("off")

plt.suptitle("One Sample per Microscope Type", fontsize=12)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/sample_images_per_microscope.png")
plt.show()

In [ ]:
# Longitudinal strip: same group across time
def plot_time_strip(df, n_timepoints=8, size=100):
    # Find a group with the most timepoints
    group_keys = ["microscope","cell_line","culture_medium","formation_method",
                  "seeding_density","magnification","biological_rep","technical_rep"]
    g = df.groupby(group_keys)
    biggest = g.size().idxmax()
    sub = g.get_group(biggest).sort_values("timepoint_hour")
    sub = sub.head(n_timepoints)

    fig, axes = plt.subplots(1, len(sub), figsize=(len(sub) * 2, 2.5))
    if len(sub) == 1: axes = [axes]
    for ax, (_, row) in zip(axes, sub.iterrows()):
        img = load_thumb(row["full_path"], size)
        ax.imshow(img, cmap="gray")
        ax.set_title(f"{int(row['timepoint_hour'])}h", fontsize=8)
        ax.axis("off")

    cell  = biggest[1]
    micro = biggest[0]
    plt.suptitle(f"Longitudinal strip — {cell} | {micro}", fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/longitudinal_strip.png")
    plt.show()

plot_time_strip(df)